## Determining the Optimal Number of Hidden Layers and Neurons for an Artificial Neural Network (ANN)

Designing the right architecture can be challenging and often requires experimentation. However, several guidelines and methods can help in making an informed decision:

### Key Strategies
- **Start Simple**  
  Begin with a simple architecture and gradually increase complexity if needed.

- **Grid Search / Random Search**  
  Use grid search or random search to try different architectures.

- **Cross-Validation**  
  Apply cross-validation to evaluate the performance of different architectures.

### Heuristics and Rules of Thumb
- The number of neurons in the hidden layer should be between the size of the input layer and the size of the output layer.
- A common practice is to start with **1–2 hidden layers**.


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from keras.models import Sequential
from keras.layers import Dense
from keras.callbacks import EarlyStopping
import pickle

In [3]:
## Load the dataset
data = pd.read_csv("../Datasets/Churn_Modelling.csv")

## Drop irrelevant columns
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

## Encode categorical variables
le = LabelEncoder()
data['Gender'] = le.fit_transform(data['Gender'])

ohe = OneHotEncoder(sparse_output=False)
geo_encoded_value = ohe.fit_transform(data[['Geography']])

## Convert to dataframe
geo_encoded_df = pd.DataFrame(data=geo_encoded_value, columns=ohe.get_feature_names_out())

## Merge this to the original data
data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)

## Divide the dataset into features and target
X = data.drop('Exited', axis=1)
y = data['Exited']

## Split the data into training and testing set
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=42)

## Scale the feature
scalar = StandardScaler()
X_train = scalar.fit_transform(X_train)
X_test = scalar.transform(X_test)
X_train = scalar.fit_transform(X_train)
X_test = scalar.transform(X_test)


## Save the encoders scalar as pickle file
with open('../artifacts/gender_label_encoder.pkl', 'wb') as file:
    pickle.dump(le, file)

with open('../artifacts/geo_onehot_encoder.pkl', 'wb') as file:
    pickle.dump(ohe, file)
 
with open('../artifacts/scalar.pkl', 'wb') as file:
    pickle.dump(scalar, file)

In [11]:
## Define a function to create a model and try different parameters

def create_model(neurons=32, layers=1):
    model=Sequential()
    model.add(Dense(neurons, activation='relu', input_shape=(X_train.shape[1],))), 

    for _ in range(layers-1):
        model.add(Dense(neurons, activation='relu'))
    
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    return model



In [13]:
## Create a kears classifier
model = KerasClassifier(neurons=32, layers=1, build_fn=create_model, epochs=50, verbose=1)

In [14]:
## Define the grid search parameters
param_grid = {
    'neurons': [16, 32, 64, 128],
    'layers': [1,2],
    # 'batch_size': [10,20],
    'epochs': [50,100]
}

In [15]:
## Perform grid search cv
grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    n_jobs=-1,
    cv=3
)

grid_result = grid.fit(X_train, y_train)

## Print the best parameters
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))

Epoch 1/100


e:\AI\03 - Deep Learning\Deep-Learning-Projects\ANN\Churn-Modelling\churn_venv\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
e:\AI\03 - Deep Learning\Deep-Learning-Projects\ANN\Churn-Modelling\churn_venv\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 948us/step - accuracy: 0.7968 - loss: 0.4694
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 941us/step - accuracy: 0.8100 - loss: 0.4395
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 960us/step - accuracy: 0.8167 - loss: 0.4289
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 973us/step - accuracy: 0.8219 - loss: 0.4201
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 949us/step - accuracy: 0.8284 - loss: 0.4103
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 937us/step - accuracy: 0.8361 - loss: 0.3995
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 958us/step - accuracy: 0.8419 - loss: 0.3883
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 954us/step - accuracy: 0.8466 - loss: 0.3787
Epoch 9/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 997us/step - accuracy: 0.8496 - loss: 0.3710
Epoch 10/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8526 - loss: 0.3652
Epoch 11/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8534 - loss: 0.3607
Epoch 12/100
250/250 ━━━━━━━━━━━━━━